In [1]:
!pip install -q langchain langchain-core langsmith

import os
from getpass import getpass
from langchain_core.runnables import RunnableLambda

os.environ["LANGSMITH_API_KEY"] = getpass("Enter LangSmith API Key: ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "AI Resume Screening Task 3"

job_skills = [
    "python", "machine learning", "data analysis", "sql", "pandas",
    "numpy", "scikit-learn", "data visualization", "statistics", "nlp"
]

job_description = """
Role: Data Scientist
Required Skills: Python, Machine Learning, Data Analysis, SQL, Pandas, NumPy,
Scikit-learn, Data Visualization, Statistics, NLP.
"""

resumes = {
    "Strong Candidate": """
    Skills: Python, Machine Learning, SQL, Pandas, NumPy, Scikit-learn,
    Data Visualization, Statistics, NLP, Deep Learning.
    Experience: 2 years building ML models and NLP projects.
    Tools: Jupyter Notebook, GitHub, Power BI, Google Colab.
    """,

    "Average Candidate": """
    Skills: Python, SQL, Pandas, Data Analysis, Basic Machine Learning.
    Experience: Internship in data cleaning and simple ML model building.
    Tools: Excel, Jupyter Notebook, Google Colab.
    """,

    "Weak Candidate": """
    Skills: HTML, CSS, JavaScript, Basic Python.
    Experience: Created frontend websites and small Python scripts.
    Tools: VS Code, GitHub.
    """
}
def extract_skills(data):
    resume = data["resume"].lower()
    found = [skill for skill in job_skills if skill in resume]
    return {
        "resume": data["resume"],
        "found_skills": found
    }
def match_skills(data):
    found = data["found_skills"]
    missing = [skill for skill in job_skills if skill not in found]
    return {
        "found_skills": found,
        "missing_skills": missing
    }
def score_candidate(data):
    found = data["found_skills"]
    missing = data["missing_skills"]

    score = int((len(found) / len(job_skills)) * 100)

    if score >= 80:
        category = "Strong Candidate"
    elif score >= 50:
        category = "Average Candidate"
    else:
        category = "Weak Candidate"

    return {
        "score": score,
        "category": category,
        "found_skills": found,
        "missing_skills": missing
    }
def explain_score(data):
    return {
        "fit_score": data["score"],
        "category": data["category"],
        "explanation": f"The candidate matched {len(data['found_skills'])} out of {len(job_skills)} required skills.",
        "matched_skills": data["found_skills"],
        "missing_skills": data["missing_skills"],
        "recommendation": "Good fit" if data["score"] >= 80 else "Needs improvement"
    }

extract_chain = RunnableLambda(extract_skills)
match_chain = RunnableLambda(match_skills)
score_chain = RunnableLambda(score_candidate)
explain_chain = RunnableLambda(explain_score)

pipeline = extract_chain | match_chain | score_chain | explain_chain

for candidate, resume in resumes.items():
    result = pipeline.invoke(
        {"resume": resume},
        config={"run_name": candidate, "tags": ["resume-screening", candidate]}
    )

    print("\n" + "="*60)
    print(candidate)
    print("="*60)
    print("Fit Score:", result["fit_score"])
    print("Category:", result["category"])
    print("Explanation:", result["explanation"])
    print("Matched Skills:", result["matched_skills"])
    print("Missing Skills:", result["missing_skills"])
    print("Recommendation:", result["recommendation"])


Enter LangSmith API Key: ··········

Strong Candidate
Fit Score: 90
Category: Strong Candidate
Explanation: The candidate matched 9 out of 10 required skills.
Matched Skills: ['python', 'machine learning', 'sql', 'pandas', 'numpy', 'scikit-learn', 'data visualization', 'statistics', 'nlp']
Missing Skills: ['data analysis']
Recommendation: Good fit

Average Candidate
Fit Score: 50
Category: Average Candidate
Explanation: The candidate matched 5 out of 10 required skills.
Matched Skills: ['python', 'machine learning', 'data analysis', 'sql', 'pandas']
Missing Skills: ['numpy', 'scikit-learn', 'data visualization', 'statistics', 'nlp']
Recommendation: Needs improvement

Weak Candidate
Fit Score: 10
Category: Weak Candidate
Explanation: The candidate matched 1 out of 10 required skills.
Matched Skills: ['python']
Missing Skills: ['machine learning', 'data analysis', 'sql', 'pandas', 'numpy', 'scikit-learn', 'data visualization', 'statistics', 'nlp']
Recommendation: Needs improvement
